# Training in Practice

Notebook for **Chapter 3** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch03.html

Tip: change a number, predict the output, then run the cell again.

## 3.1 Loss Functions: Defining "Wrong"

In [ ]:
import numpy as np
print("MSE", np.mean((np.array([2.5,0.0,4.0])-np.array([3.0,0.5,5.0]))**2))
p = np.array([0.7,0.2,0.1]); print("CE true=0", -np.log(p[0]))
for pt in [0.9,0.1]:
    ce = -np.log(pt); print("p_t", pt, "CE", round(ce,4), "focal g=2", round((1-pt)**2*ce,5))
d = 0.4; m = 1.0
print("contrastive similar", 0.5*d**2, "dissimilar", 0.5*max(0,m-d)**2)
print("triplet", max(0,0.5-0.8+0.5), max(0,0.5-0.2+0.5))

MSE 0.5
CE true=0 0.35667494393873245
p_t 0.9 CE 0.1054 focal g=2 0.00105
p_t 0.1 CE 2.3026 focal g=2 1.86509
contrastive similar 0.08000000000000002 dissimilar 0.18
triplet 0.19999999999999996 0.8


## 3.2 Optimizers and Learning-Rate Schedules

In [ ]:
grad = lambda w: np.array([2*w[0], 20*w[1]])
f = lambda w: w[0]**2 + 10*w[1]**2
def run(name, steps=30):
    w = np.array([1.,1.]); v = np.zeros(2); s = np.zeros(2); m = np.zeros(2); vv = np.zeros(2)
    for t in range(1, steps+1):
        g = grad(w)
        if name == "sgd":  w = w - 0.09*g
        if name == "mom":  v = 0.9*v + g; w = w - 0.02*v
        if name == "rms":  s = 0.9*s + 0.1*g*g; w = w - 0.05*g/(np.sqrt(s)+1e-8)
        if name == "adam":
            m = 0.9*m + 0.1*g; vv = 0.999*vv + 0.001*g*g
            mh = m/(1-0.9**t); vh = vv/(1-0.999**t); w = w - 0.1*mh/(np.sqrt(vh)+1e-8)
    return w, f(w)
for n in ["sgd","mom","rms","adam"]: print(n, run(n))

sgd (array([0.00259666, 0.00123794]), np.float64(2.2067613579465364e-05))
mom (array([0.1851202 , 0.08200961]), np.float64(0.10152525663190662))
rms (array([0.00584382, 0.00584382]), np.float64(0.000375652525585838))
adam (array([-0.03396752, -0.03396752]), np.float64(0.012691714688971227))


In [ ]:
def sched(t, T=100, warm=10, peak=1e-3):
    if t < warm: return peak*(t+1)/warm
    return 0.5*peak*(1+np.cos(np.pi*(t-warm)/(T-warm)))
print([round(float(sched(t)), 6) for t in [0,4,9,10,30,55,99]])

[0.0001, 0.0005, 0.001, 0.001, 0.000883, 0.0005, 0.0]


## 3.3 Initialization and Normalization

In [ ]:
rng = np.random.default_rng(0)
x0 = rng.normal(size=(512,256))
def deep(act, scale, L=10):
    x = x0; out = []
    for l in range(L):
        W = rng.normal(size=(256,256))*scale(256)
        x = act(x@W); out.append(x.std())
    return np.array(out)
relu = lambda z: np.maximum(0,z)
print("tanh std0.01", deep(np.tanh, lambda n: 0.01)[[0,4,9]])
print("tanh xavier ", deep(np.tanh, lambda n: np.sqrt(1/n))[[0,4,9]])
print("relu xavier ", deep(relu,    lambda n: np.sqrt(1/n))[[0,4,9]])
print("relu he     ", deep(relu,    lambda n: np.sqrt(2/n))[[0,4,9]])

tanh std0.01 [0.1564 0.0001 0.    ]
tanh xavier  [0.6287 0.3232 0.2262]
relu xavier  [0.5871 0.1322 0.0282]
relu he      [0.8253 0.8854 0.7753]


In [ ]:
X = np.array([[1.,2,3],[2,4,6],[3,6,12],[0,0,0]]); eps = 1e-5
bn  = (X-X.mean(0))/np.sqrt(X.var(0)+eps)
ln  = (X-X.mean(1,keepdims=True))/np.sqrt(X.var(1,keepdims=True)+eps)
rms = X/np.sqrt((X**2).mean(1,keepdims=True)+eps)
print("BN\n",bn); print("LN\n",ln); print("RMS\n",rms)
c = np.array([1.,3,10,14]); g = c.reshape(2,2)
print("GN", ((g-g.mean(1,keepdims=True))/np.sqrt(g.var(1,keepdims=True)+eps)).ravel())

BN
 [[-0.4472 -0.4472 -0.5071]
 [ 0.4472  0.4472  0.169 ]
 [ 1.3416  1.3416  1.5213]
 [-1.3416 -1.3416 -1.1832]]
LN
 [[-1.2247  0.      1.2247]
 [-1.2247  0.      1.2247]
 [-1.069  -0.2673  1.3363]
 [ 0.      0.      0.    ]]
RMS
 [[0.4629 0.9258 1.3887]
 [0.4629 0.9258 1.3887]
 [0.378  0.7559 1.5119]
 [0.     0.     0.    ]]
GN [-1. 1. -1. 1.]


## 3.4 Regularization: Stopping the Model from Memorizing

In [ ]:
w = np.array([2.0,-0.5]); lr = 0.1; lam = 0.1
print("L2 step", w-lr*lam*w, "L1 step", w-lr*lam*np.sign(w))
r = np.random.default_rng(1); h = np.array([1.,2,3,4,5,6]); mask = (r.random(6) > 0.5)
print("mask", mask.astype(int), "dropped", h*mask/0.5)
K = 3; t = np.array([1,0,0])*(1-0.1) + 0.1/K
print("smooth target", t, "loss", -(t*np.log(np.array([0.7,0.2,0.1]))).sum())
print("mixup x", 0.7*np.array([1.,0]) + 0.3*np.array([0.,1]))

L2 step [ 1.98  -0.495] L1 step [ 1.99 -0.49]
mask [1 1 0 1 0 0] dropped [2. 4. 0. 8. 0. 0.]
smooth target [0.9333 0.0333 0.0333] loss 0.46329738119042185
mixup x [0.7 0.3]


## 3.5 Going Further: Four Norms on One Tensor, Decoupled Weight Decay, Schedules and Label Smoothing

Four ideas in this chapter are easy to say and easy to mix up: which normalisation does what, why AdamW exists, what shape a learning-rate schedule has, and what label smoothing does to the loss. Here each gets a fresh numeric example you can check by hand.

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
# batch of 2 samples, 4 channels (features)
X = np.array([[1., 2., 3., 6.],
              [3., 2., 1., 0.]])
eps = 1e-5
def norm(x, axis): 
    m = x.mean(axis=axis, keepdims=True); v = x.var(axis=axis, keepdims=True)
    return (x-m)/np.sqrt(v+eps)
print("BatchNorm (per feature, over the batch)\n", norm(X, 0))
print("LayerNorm (per sample, over features)\n", norm(X, 1))
G = X.reshape(2, 2, 2)        # 2 groups of 2 channels
gn = norm(G, 2).reshape(2, 4)
print("GroupNorm, 2 groups of 2 channels\n", gn)
rms = X/np.sqrt((X**2).mean(axis=1, keepdims=True)+eps)
print("RMSNorm (per sample, no mean removal)\n", rms)
print("row means after LayerNorm", norm(X,1).mean(1).round(4), " after RMSNorm", rms.mean(1).round(4))

BatchNorm (per feature, over the batch)
 [[-1.  0.  1.  1.]
 [ 1.  0. -1. -1.]]
LayerNorm (per sample, over features)
 [[-1.069  -0.5345  0.      1.6036]
 [ 1.3416  0.4472 -0.4472 -1.3416]]
GroupNorm, 2 groups of 2 channels
 [[-1.  1. -1.  1.]
 [ 1. -1.  1. -1.]]
RMSNorm (per sample, no mean removal)
 [[0.2828 0.5657 0.8485 1.6971]
 [1.6036 1.069  0.5345 0.    ]]
row means after LayerNorm [-0.  0.]  after RMSNorm [0.8485 0.8018]


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
def run(decoupled, gscale, steps=1000, lr=1e-3, wd=1.0, b1=0.9, b2=0.999, eps=1e-8):
    w, m, v = 1.0, 0.0, 0.0
    for t in range(1, steps+1):
        g = gscale*rng.normal()                  # loss gradient: pure noise, mean zero
        if not decoupled: g = g + wd*w           # Adam + L2: decay enters the gradient
        m = b1*m + (1-b1)*g; v = b2*v + (1-b2)*g*g
        mh, vh = m/(1-b1**t), v/(1-b2**t)
        w -= lr*mh/(np.sqrt(vh)+eps)
        if decoupled: w -= lr*wd*w               # AdamW: decay applied directly to the weight
    return w
for gs in [0.01, 1.0, 100.0]:
    print(f"gradient noise scale {gs:>6}: Adam+L2 final w = {run(False, gs):.3f}   AdamW final w = {run(True, gs):.3f}")
print("pure decay (1 - lr*wd)^1000 =", round((1-1e-3*1.0)**1000, 3))

gradient noise scale   0.01: Adam+L2 final w = 0.258   AdamW final w = 0.384
gradient noise scale    1.0: Adam+L2 final w = 0.480   AdamW final w = 0.343
gradient noise scale  100.0: Adam+L2 final w = 0.943   AdamW final w = 0.352
pure decay (1 - lr*wd)^1000 = 0.368


In [ ]:
import numpy as np
peak, floor, total, warm = 1e-3, 1e-4, 100, 10
def cosine(t):
    if t < warm: return peak*(t+1)/warm
    pr = (t-warm)/(total-warm)
    return floor + 0.5*(peak-floor)*(1+np.cos(np.pi*pr))
def linear(t):
    if t < warm: return peak*(t+1)/warm
    pr = (t-warm)/(total-warm)
    return peak + (floor-peak)*pr
def step(t):
    return peak*(0.1**(t//40))
for t in [0, 9, 10, 40, 55, 99]:
    print(f"step {t:>2}: cosine {cosine(t):.6f}  linear {linear(t):.6f}  step-decay {step(t):.6f}")

step  0: cosine 0.000100  linear 0.000100  step-decay 0.001000
step  9: cosine 0.001000  linear 0.001000  step-decay 0.001000
step 10: cosine 0.001000  linear 0.001000  step-decay 0.001000
step 40: cosine 0.000775  linear 0.000700  step-decay 0.000100
step 55: cosine 0.000550  linear 0.000550  step-decay 0.000100
step 99: cosine 0.000100  linear 0.000110  step-decay 0.000010


In [ ]:
import numpy as np
K, eps = 4, 0.1
y = 0
smooth = np.full(K, eps/K); smooth[y] += 1-eps
print("smoothed target:", smooth)
logits = np.array([4.0, 0.0, 0.0, 0.0])
p = np.exp(logits-logits.max()); p/=p.sum()
print("probs:", np.round(p,4))
print("hard CE  :", round(-np.log(p[y]),4))
print("smooth CE:", round(-(smooth*np.log(p)).sum(),4))
gap = np.log(0.925/0.025); print("optimal gap:", round(gap,4))
K=10; t=np.full(K,0.1/K); t[0]+=0.9; print("K=10 target:", t[:2], t.sum())

smoothed target: [0.925 0.025 0.025 0.025]
probs: [0.9479 0.0174 0.0174 0.0174]
hard CE  : 0.0535
smooth CE: 0.3535
optimal gap: 3.6109
K=10 target: [0.91 0.01] 1.0


## Chapter recap

## Practice set

**1.** (easy) The model gives the true class probability 0.25. What is CE? **Answer:** `−ln 0.25 = 1.3863`.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch03.html